# Project 5 — Deep Learning Chemistry

This notebook turns the earlier classical cheminformatics projects into deep-learning practice.

It contains three independent but connected versions:

| Version | Representation | Model | Main task |
|---|---|---|---|
| Version 1 | Morgan/ECFP fingerprints | PyTorch MLP | ESOL logS regression |
| Version 2 | Molecular graph | PyTorch Geometric GNN | High/low solubility classification |
| Version 3A | DeepChem ECFP | Random Forest / Extra Trees | ESOL regression baseline |
| Version 3B | DeepChem ECFP | Random Forest / Extra Trees | Tox21 classification baseline |
| Version 3C | DeepChem ECFP | PyTorch MLP | ESOL regression with DeepChem split |

The purpose is not to claim that deep learning always wins. The purpose is to practise molecular representation, neural-network training, graph learning, benchmark loaders, diagnostics and scientific interpretation.

## Chapter 0 — Setup and learning goals

By the end of this notebook, you should understand:

- how a SMILES string becomes a neural-network input;
- how Morgan fingerprints differ from molecular graphs;
- how to train a PyTorch MLP for molecular regression;
- how to train a PyTorch Geometric GNN for molecular classification;
- how to use DeepChem as a dataset/benchmark framework;
- how to compare neural-network models against strong classical baselines.

Recommended environment:

```bash
conda install -c conda-forge rdkit pandas numpy scikit-learn matplotlib pytorch
pip install torch-geometric deepchem
```

`torch-geometric` and `deepchem` are optional sections. The notebook will tell you if they are unavailable.

In [ ]:
# Core imports
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors, Draw
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

RANDOM_STATE = 42


def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_STATE)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## Chapter 1 — Shared ESOL dataset

Most examples use the Delaney/ESOL dataset. It contains molecular structures and measured aqueous solubility values.

Target meaning:

```text
logS = measured log solubility in mol/L
```

For regression, we predict exact `logS`. For classification, we create a high/low solubility label using the median logS value.

In [ ]:
ESOL_URL = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
LOCAL_ESOL_PATH = Path("../data/delaney-processed.csv")


def load_esol_dataframe(url=ESOL_URL, local_path=LOCAL_ESOL_PATH):
    """Load ESOL from a local file if available; otherwise load from the public URL."""
    if local_path.exists():
        raw_df = pd.read_csv(local_path)
        print(f"Loaded local ESOL file: {local_path}")
    else:
        raw_df = pd.read_csv(url)
        print("Loaded ESOL from public URL")

    print("Raw shape:", raw_df.shape)
    print("Raw columns:", raw_df.columns.tolist())
    return raw_df


raw_df = load_esol_dataframe()
raw_df.head()

In [ ]:
# Standardise column names used in this notebook.
esol_df = raw_df.rename(columns={
    "Compound ID": "name",
    "measured log solubility in mols per litre": "target",
}).copy()

required_columns = ["name", "smiles", "target"]
missing = [col for col in required_columns if col not in esol_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

esol_df = esol_df[required_columns].dropna().reset_index(drop=True)
esol_df.head()

In [ ]:
def smiles_to_mol(smiles):
    """Convert a SMILES string into an RDKit molecule, returning None for invalid input."""
    if pd.isna(smiles):
        return None
    try:
        return Chem.MolFromSmiles(str(smiles))
    except Exception:
        return None


def canonical_smiles(mol):
    """Return canonical SMILES for a valid RDKit molecule."""
    return Chem.MolToSmiles(mol, canonical=True)


esol_df["mol"] = esol_df["smiles"].apply(smiles_to_mol)

before = len(esol_df)
esol_df = esol_df[esol_df["mol"].notnull()].reset_index(drop=True)
print("Invalid SMILES removed:", before - len(esol_df))

esol_df["canonical_smiles"] = esol_df["mol"].apply(canonical_smiles)

before = len(esol_df)
esol_df = esol_df.drop_duplicates(subset="canonical_smiles").reset_index(drop=True)
print("Duplicate molecules removed:", before - len(esol_df))

# Binary label for classification examples: 1 = above median solubility, 0 = below/equal median solubility.
threshold = esol_df["target"].median()
esol_df["solubility_class"] = (esol_df["target"] > threshold).astype(int)

print("Clean shape:", esol_df.shape)
print("Median logS threshold:", threshold)
print(esol_df["solubility_class"].value_counts())
esol_df.head()

## Chapter 2 — Descriptor sanity checks

Before deep learning, compute a few interpretable RDKit descriptors. These are useful for checking whether the neural-network predictions make chemical sense.

In [ ]:
def calculate_basic_descriptors(mol):
    """Calculate compact molecular descriptors for interpretation and sanity checks."""
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotBonds": Lipinski.NumRotatableBonds(mol),
        "RingCount": Lipinski.RingCount(mol),
        "HeavyAtomCount": Lipinski.HeavyAtomCount(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
    }


desc_df = pd.DataFrame([calculate_basic_descriptors(mol) for mol in esol_df["mol"]])
esol_with_desc_df = pd.concat([esol_df.reset_index(drop=True), desc_df.reset_index(drop=True)], axis=1)
esol_with_desc_df.head()

In [ ]:
# Quick relationship check between descriptors and logS.
descriptor_cols = ["MolWt", "LogP", "TPSA", "HBD", "HBA", "RotBonds", "RingCount", "HeavyAtomCount", "FractionCSP3"]

corr = esol_with_desc_df[descriptor_cols + ["target"]].corr(numeric_only=True)["target"].sort_values()
corr

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(esol_with_desc_df["LogP"], esol_with_desc_df["target"], alpha=0.6)
plt.xlabel("RDKit MolLogP")
plt.ylabel("Measured logS")
plt.title("Hydrophobicity vs solubility")
plt.show()

## Chapter 3 — Version 1: Morgan fingerprints + PyTorch MLP regression

This is the first deep-learning version.

Pipeline:

```text
SMILES -> RDKit Mol -> Morgan fingerprint vector -> PyTorch MLP -> predicted logS
```

This is still fingerprint-based QSAR, but the model is a neural network rather than Random Forest, Extra Trees or SVR.

In [ ]:
FP_SIZE = 2048
RADIUS = 2
morgan_generator = GetMorganGenerator(radius=RADIUS, fpSize=FP_SIZE)


def mol_to_morgan_fp(mol):
    """Convert an RDKit molecule to a dense Morgan fingerprint array."""
    fp = morgan_generator.GetFingerprint(mol)
    return np.array(fp, dtype=np.float32)


X_fp = np.stack(esol_df["mol"].apply(mol_to_morgan_fp).values)
y_logS = esol_df["target"].values.astype(np.float32)

print("Fingerprint matrix:", X_fp.shape)
print("Target vector:", y_logS.shape)
print("Fingerprint density:", X_fp.mean())

In [ ]:
# Train/validation/test split for the manual PyTorch MLP.
X_temp, X_test, y_temp, y_test = train_test_split(
    X_fp,
    y_logS,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp,
    y_temp,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test:", X_test.shape)

In [ ]:
class MoleculeRegressionDataset(Dataset):
    """Torch dataset for molecular fingerprint regression."""

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = MoleculeRegressionDataset(X_train, y_train)
valid_ds = MoleculeRegressionDataset(X_valid, y_valid)
test_ds = MoleculeRegressionDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

In [ ]:
class FingerprintMLPRegressor(nn.Module):
    """MLP regressor for ECFP/Morgan fingerprints."""

    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.40),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.network(x)


v1_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
v1_model = FingerprintMLPRegressor(input_dim=X_train.shape[1]).to(v1_device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(v1_model.parameters(), lr=5e-4, weight_decay=1e-3)

print(v1_model)
print("Device:", v1_device)

In [ ]:
def evaluate_mse_loss(model, loader, device):
    """Return average MSE loss for a regression model."""
    model.eval()
    total_loss = 0.0
    total_n = 0
    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            preds = model(batch_X)
            loss = criterion(preds, batch_y)
            total_loss += loss.item() * batch_X.size(0)
            total_n += batch_X.size(0)
    return total_loss / total_n


def predict_regression(model, loader, device):
    """Return true and predicted values from a regression model."""
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)
            preds = model(batch_X).cpu().numpy().ravel()
            all_pred.extend(preds)
            all_true.extend(batch_y.numpy().ravel())
    return np.array(all_true), np.array(all_pred)


def regression_metrics(y_true, y_pred):
    """Return standard regression metrics."""
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    }

In [ ]:
# Train the fingerprint MLP with early stopping.
n_epochs = 100
patience = 15
best_valid_loss = np.inf
best_state = None
patience_counter = 0

v1_train_losses = []
v1_valid_losses = []

for epoch in range(n_epochs):
    v1_model.train()
    running_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(v1_device)
        batch_y = batch_y.to(v1_device)

        optimizer.zero_grad()
        preds = v1_model(batch_X)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_X.size(0)

    train_loss = running_loss / len(train_ds)
    valid_loss = evaluate_mse_loss(v1_model, valid_loader, v1_device)

    v1_train_losses.append(train_loss)
    v1_valid_losses.append(valid_loss)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = {k: v.detach().cpu().clone() for k, v in v1_model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:03d} | Train MSE {train_loss:.4f} | Valid MSE {valid_loss:.4f}")

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch + 1}")
        break

if best_state is not None:
    v1_model.load_state_dict(best_state)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(v1_train_losses, label="Train")
plt.plot(v1_valid_losses, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Version 1 MLP training curve")
plt.legend()
plt.show()

In [ ]:
y_train_true, y_train_pred = predict_regression(v1_model, train_loader, v1_device)
y_valid_true, y_valid_pred = predict_regression(v1_model, valid_loader, v1_device)
y_test_true, y_test_pred = predict_regression(v1_model, test_loader, v1_device)

v1_metrics_df = pd.DataFrame([
    {"split": "train", **regression_metrics(y_train_true, y_train_pred)},
    {"split": "valid", **regression_metrics(y_valid_true, y_valid_pred)},
    {"split": "test", **regression_metrics(y_test_true, y_test_pred)},
])

v1_metrics_df

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test_true, y_test_pred, alpha=0.7)
min_value = min(y_test_true.min(), y_test_pred.min())
max_value = max(y_test_true.max(), y_test_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.xlabel("Actual logS")
plt.ylabel("Predicted logS")
plt.title("Version 1 MLP: predicted vs actual")
plt.show()

residuals = y_test_true - y_test_pred
plt.figure(figsize=(7, 5))
plt.scatter(y_test_pred, residuals, alpha=0.7)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted logS")
plt.ylabel("Residual: actual - predicted")
plt.title("Version 1 MLP residual plot")
plt.show()

### Version 1 interpretation checklist

Use the output above to decide whether the MLP is useful:

- If train loss is much lower than validation loss, the MLP is overfitting.
- If test RMSE is similar to or better than the tree baselines in Version 3A, the neural network is competitive.
- If predictions are compressed toward the mean, the model is ranking molecules but not estimating extreme solubility well.

In [ ]:
# Predict logS for a few new molecules.
new_molecules = pd.DataFrame({
    "name": ["benzyl acetate", "ambroxide", "glucose", "limonene", "vanillin"],
    "smiles": [
        "CC(=O)OCC1=CC=CC=C1",
        "CC1(C)CCCC2(C)C1CCC(O2)C(C)(C)O",
        "C(C1C(C(C(C(O1)O)O)O)O)O",
        "CC1=CCC(CC1)C(=C)C",
        "COC1=C(C=CC(=C1)C=O)O",
    ],
})

new_molecules["mol"] = new_molecules["smiles"].apply(smiles_to_mol)
new_molecules = new_molecules[new_molecules["mol"].notnull()].reset_index(drop=True)
new_X = np.stack(new_molecules["mol"].apply(mol_to_morgan_fp).values)
new_loader = DataLoader(MoleculeRegressionDataset(new_X, np.zeros(len(new_X))), batch_size=64, shuffle=False)
_, new_pred = predict_regression(v1_model, new_loader, v1_device)
new_molecules["predicted_logS"] = new_pred
new_molecules[["name", "smiles", "predicted_logS"]]

## Chapter 4 — Version 2: Molecular graph + PyTorch Geometric GNN

Version 1 used a fixed fingerprint vector. Version 2 represents each molecule as a graph.

Pipeline:

```text
SMILES -> RDKit Mol -> atoms as nodes + bonds as edges -> GNN -> prediction
```

This chapter uses a binary classification task:

```text
0 = lower-than-median ESOL solubility
1 = higher-than-median ESOL solubility
```

This gives a stable graph-classification practice task. Exact logS regression can be added later by changing the final loss to `MSELoss`.

In [ ]:
try:
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader as PyGDataLoader
    from torch_geometric.nn import GCNConv, global_mean_pool
    PYG_AVAILABLE = True
except Exception as exc:
    PYG_AVAILABLE = False
    print("PyTorch Geometric is not available. Install with: pip install torch-geometric")
    print("Import error:", exc)

In [ ]:
if PYG_AVAILABLE:
    def atom_features(atom):
        """Simple numeric atom features for a beginner GNN."""
        return [
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            atom.GetTotalNumHs(),
            int(atom.GetHybridization()),
            int(atom.GetIsAromatic()),
            int(atom.IsInRing()),
        ]

    def smiles_to_graph(smiles, label):
        """Convert SMILES into a PyTorch Geometric Data graph."""
        mol = smiles_to_mol(smiles)
        if mol is None:
            return None

        x = torch.tensor([atom_features(atom) for atom in mol.GetAtoms()], dtype=torch.float32)

        edge_pairs = []
        for bond in mol.GetBonds():
            start = bond.GetBeginAtomIdx()
            end = bond.GetEndAtomIdx()
            edge_pairs.append([start, end])
            edge_pairs.append([end, start])

        if edge_pairs:
            edge_index = torch.tensor(edge_pairs, dtype=torch.long).t().contiguous()
        else:
            edge_index = torch.empty((2, 0), dtype=torch.long)

        y = torch.tensor([label], dtype=torch.float32)
        return Data(x=x, edge_index=edge_index, y=y)

    graph_list = []
    for _, row in esol_df.iterrows():
        graph = smiles_to_graph(row["canonical_smiles"], row["solubility_class"])
        if graph is not None:
            graph.name = row["name"]
            graph.smiles = row["canonical_smiles"]
            graph.target_logS = float(row["target"])
            graph_list.append(graph)

    print("Number of molecular graphs:", len(graph_list))
    print("Example graph:", graph_list[0])

In [ ]:
if PYG_AVAILABLE:
    labels = np.array([int(graph.y.item()) for graph in graph_list])

    train_graphs, test_graphs = train_test_split(
        graph_list,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=labels,
    )

    train_labels = np.array([int(graph.y.item()) for graph in train_graphs])
    train_graphs, valid_graphs = train_test_split(
        train_graphs,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=train_labels,
    )

    pyg_train_loader = PyGDataLoader(train_graphs, batch_size=64, shuffle=True)
    pyg_valid_loader = PyGDataLoader(valid_graphs, batch_size=64, shuffle=False)
    pyg_test_loader = PyGDataLoader(test_graphs, batch_size=64, shuffle=False)

    print("Train graphs:", len(train_graphs))
    print("Validation graphs:", len(valid_graphs))
    print("Test graphs:", len(test_graphs))

In [ ]:
if PYG_AVAILABLE:
    class MoleculeGCNClassifier(nn.Module):
        """Small GCN for graph-level molecular classification."""

        def __init__(self, num_node_features, hidden_dim=64):
            super().__init__()
            self.conv1 = GCNConv(num_node_features, hidden_dim)
            self.conv2 = GCNConv(hidden_dim, hidden_dim)
            self.conv3 = GCNConv(hidden_dim, hidden_dim)
            self.classifier = nn.Sequential(
                nn.Linear(hidden_dim, 64),
                nn.ReLU(),
                nn.Dropout(0.30),
                nn.Linear(64, 1),
            )

        def forward(self, x, edge_index, batch):
            x = torch.relu(self.conv1(x, edge_index))
            x = torch.relu(self.conv2(x, edge_index))
            x = torch.relu(self.conv3(x, edge_index))
            graph_embedding = global_mean_pool(x, batch)
            return self.classifier(graph_embedding)

    gnn_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    gnn_model = MoleculeGCNClassifier(num_node_features=graph_list[0].x.shape[1], hidden_dim=64).to(gnn_device)
    gnn_criterion = nn.BCEWithLogitsLoss()
    gnn_optimizer = torch.optim.AdamW(gnn_model.parameters(), lr=1e-3, weight_decay=1e-4)

    print(gnn_model)
    print("Device:", gnn_device)

In [ ]:
if PYG_AVAILABLE:
    def evaluate_gnn_loss(model, loader):
        model.eval()
        total_loss = 0.0
        total_graphs = 0
        with torch.no_grad():
            for batch in loader:
                batch = batch.to(gnn_device)
                logits = model(batch.x, batch.edge_index, batch.batch)
                y = batch.y.view(-1, 1).float()
                loss = gnn_criterion(logits, y)
                total_loss += loss.item() * batch.num_graphs
                total_graphs += batch.num_graphs
        return total_loss / total_graphs

    def predict_gnn_classifier(model, loader):
        model.eval()
        all_probs, all_pred, all_true = [], [], []
        with torch.no_grad():
            for batch in loader:
                batch = batch.to(gnn_device)
                logits = model(batch.x, batch.edge_index, batch.batch)
                probs = torch.sigmoid(logits).cpu().numpy().ravel()
                pred = (probs >= 0.5).astype(int)
                true = batch.y.cpu().numpy().ravel().astype(int)
                all_probs.extend(probs)
                all_pred.extend(pred)
                all_true.extend(true)
        return np.array(all_true), np.array(all_pred), np.array(all_probs)

In [ ]:
if PYG_AVAILABLE:
    n_epochs = 100
    patience = 15
    best_valid_loss = np.inf
    best_state = None
    patience_counter = 0

    gnn_train_losses = []
    gnn_valid_losses = []

    for epoch in range(n_epochs):
        gnn_model.train()
        running_loss = 0.0

        for batch in pyg_train_loader:
            batch = batch.to(gnn_device)
            gnn_optimizer.zero_grad()
            logits = gnn_model(batch.x, batch.edge_index, batch.batch)
            y = batch.y.view(-1, 1).float()
            loss = gnn_criterion(logits, y)
            loss.backward()
            gnn_optimizer.step()
            running_loss += loss.item() * batch.num_graphs

        train_loss = running_loss / len(train_graphs)
        valid_loss = evaluate_gnn_loss(gnn_model, pyg_valid_loader)

        gnn_train_losses.append(train_loss)
        gnn_valid_losses.append(valid_loss)

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = {k: v.detach().cpu().clone() for k, v in gnn_model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1:03d} | Train loss {train_loss:.4f} | Valid loss {valid_loss:.4f}")

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    if best_state is not None:
        gnn_model.load_state_dict(best_state)

In [ ]:
if PYG_AVAILABLE:
    plt.figure(figsize=(7, 5))
    plt.plot(gnn_train_losses, label="Train")
    plt.plot(gnn_valid_losses, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("BCE loss")
    plt.title("Version 2 GNN training curve")
    plt.legend()
    plt.show()

In [ ]:
if PYG_AVAILABLE:
    y_true, y_pred, y_prob = predict_gnn_classifier(gnn_model, pyg_test_loader)

    gnn_results = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
    }

    display(pd.DataFrame([gnn_results]))
    print(classification_report(y_true, y_pred, zero_division=0))

    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=["Low solubility", "High solubility"])
    plt.title("Version 2 GNN confusion matrix")
    plt.show()

    RocCurveDisplay.from_predictions(y_true, y_prob)
    plt.title("Version 2 GNN ROC curve")
    plt.show()

### Version 2 interpretation checklist

A useful GNN should show:

- validation loss decreasing or stabilising;
- ROC-AUC clearly above 0.5;
- confusion matrix with both classes detected;
- fewer errors after increasing data and improving atom/bond features.

If the GNN performs worse than the fingerprint MLP or tree baseline, that is not a failure. It means the graph model needs more data, better features, bond attributes, or a stronger architecture such as GIN/MPNN.

## Chapter 5 — Version 3A: DeepChem ECFP + sklearn regression baseline

DeepChem is useful as a molecular ML framework because it provides:

- MoleculeNet dataset loaders;
- molecular featurisers;
- scaffold splitters;
- benchmark-style datasets and metrics.

This section uses DeepChem for data loading and ECFP featurisation, then trains sklearn regressors. This is not deep learning, but it gives a strong benchmark.

In [ ]:
try:
    import deepchem as dc
    DEEPCHEM_AVAILABLE = True
    print("DeepChem:", dc.__version__)
except Exception as exc:
    DEEPCHEM_AVAILABLE = False
    print("DeepChem is not available. Install with: pip install deepchem")
    print("Import error:", exc)

In [ ]:
def rmse_score(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5


def evaluate_regressor(model, X_train, y_train, X_valid, y_valid, X_test, y_test):
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_valid = model.predict(X_valid)
    pred_test = model.predict(X_test)
    row = {
        "train_MAE": mean_absolute_error(y_train, pred_train),
        "train_RMSE": rmse_score(y_train, pred_train),
        "train_R2": r2_score(y_train, pred_train),
        "valid_MAE": mean_absolute_error(y_valid, pred_valid),
        "valid_RMSE": rmse_score(y_valid, pred_valid),
        "valid_R2": r2_score(y_valid, pred_valid),
        "test_MAE": mean_absolute_error(y_test, pred_test),
        "test_RMSE": rmse_score(y_test, pred_test),
        "test_R2": r2_score(y_test, pred_test),
    }
    return row, pred_test

In [ ]:
if DEEPCHEM_AVAILABLE:
    dc_tasks, dc_datasets, dc_transformers = dc.molnet.load_delaney(
        featurizer="ECFP",
        splitter="scaffold",
    )

    dc_train, dc_valid, dc_test = dc_datasets

    X_train_dc = dc_train.X
    y_train_dc = dc_train.y.ravel()
    X_valid_dc = dc_valid.X
    y_valid_dc = dc_valid.y.ravel()
    X_test_dc = dc_test.X
    y_test_dc = dc_test.y.ravel()

    print("Tasks:", dc_tasks)
    print("Train:", X_train_dc.shape)
    print("Validation:", X_valid_dc.shape)
    print("Test:", X_test_dc.shape)

In [ ]:
if DEEPCHEM_AVAILABLE:
    baseline_models = {
        "Random Forest": RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1),
        "Extra Trees": ExtraTreesRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1),
    }

    baseline_rows = []
    baseline_predictions = {}

    for model_name, model in baseline_models.items():
        row, pred_test = evaluate_regressor(
            model,
            X_train_dc,
            y_train_dc,
            X_valid_dc,
            y_valid_dc,
            X_test_dc,
            y_test_dc,
        )
        row["model"] = model_name
        baseline_rows.append(row)
        baseline_predictions[model_name] = pred_test

    baseline_df = pd.DataFrame(baseline_rows)[[
        "model",
        "train_MAE", "train_RMSE", "train_R2",
        "valid_MAE", "valid_RMSE", "valid_R2",
        "test_MAE", "test_RMSE", "test_R2",
    ]]

    display(baseline_df)

In [ ]:
if DEEPCHEM_AVAILABLE:
    best_baseline_name = baseline_df.sort_values("test_RMSE").iloc[0]["model"]
    best_baseline_pred = baseline_predictions[best_baseline_name]

    plt.figure(figsize=(6, 6))
    plt.scatter(y_test_dc, best_baseline_pred, alpha=0.7)
    min_value = min(y_test_dc.min(), best_baseline_pred.min())
    max_value = max(y_test_dc.max(), best_baseline_pred.max())
    plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
    plt.xlabel("Actual logS")
    plt.ylabel("Predicted logS")
    plt.title(f"DeepChem ECFP + {best_baseline_name}")
    plt.show()

    residuals = y_test_dc - best_baseline_pred
    plt.figure(figsize=(7, 5))
    plt.scatter(best_baseline_pred, residuals, alpha=0.7)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted logS")
    plt.ylabel("Residual: actual - predicted")
    plt.title(f"Residual plot: {best_baseline_name}")
    plt.show()

## Chapter 6 — Version 3B: DeepChem Tox21 classification baseline

Tox21 is a multi-task toxicity dataset. A molecule can have multiple assay labels, and some labels are missing.

This section selects one assay first. That makes the workflow easier to inspect before moving to true multi-task modelling.

In [ ]:
if DEEPCHEM_AVAILABLE:
    tox_tasks, tox_datasets, tox_transformers = dc.molnet.load_tox21(
        featurizer="ECFP",
        splitter="scaffold",
    )

    tox_train, tox_valid, tox_test = tox_datasets

    print("Number of tasks:", len(tox_tasks))
    for i, task in enumerate(tox_tasks):
        print(i, task)

    print("Train X:", tox_train.X.shape)
    print("Train y:", tox_train.y.shape)
    print("Train w:", tox_train.w.shape)

In [ ]:
if DEEPCHEM_AVAILABLE:
    task_index = 0
    task_name = tox_tasks[task_index]

    X_train_tox = tox_train.X
    y_train_tox = tox_train.y[:, task_index]
    w_train_tox = tox_train.w[:, task_index]

    X_valid_tox = tox_valid.X
    y_valid_tox = tox_valid.y[:, task_index]
    w_valid_tox = tox_valid.w[:, task_index]

    X_test_tox = tox_test.X
    y_test_tox = tox_test.y[:, task_index]
    w_test_tox = tox_test.w[:, task_index]

    train_mask = w_train_tox > 0
    valid_mask = w_valid_tox > 0
    test_mask = w_test_tox > 0

    X_train_tox = X_train_tox[train_mask]
    y_train_tox = y_train_tox[train_mask].astype(int)
    X_valid_tox = X_valid_tox[valid_mask]
    y_valid_tox = y_valid_tox[valid_mask].astype(int)
    X_test_tox = X_test_tox[test_mask]
    y_test_tox = y_test_tox[test_mask].astype(int)

    print("Selected task:", task_name)
    print("Train labels:", np.bincount(y_train_tox))
    print("Validation labels:", np.bincount(y_valid_tox))
    print("Test labels:", np.bincount(y_test_tox))

In [ ]:
def evaluate_classifier(model, X_train, y_train, X_valid, y_valid, X_test, y_test):
    model.fit(X_train, y_train)
    valid_prob = model.predict_proba(X_valid)[:, 1]
    test_prob = model.predict_proba(X_test)[:, 1]
    valid_pred = (valid_prob >= 0.5).astype(int)
    test_pred = (test_prob >= 0.5).astype(int)

    row = {
        "valid_accuracy": accuracy_score(y_valid, valid_pred),
        "valid_precision": precision_score(y_valid, valid_pred, zero_division=0),
        "valid_recall": recall_score(y_valid, valid_pred, zero_division=0),
        "valid_F1": f1_score(y_valid, valid_pred, zero_division=0),
        "valid_ROC_AUC": roc_auc_score(y_valid, valid_prob),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_precision": precision_score(y_test, test_pred, zero_division=0),
        "test_recall": recall_score(y_test, test_pred, zero_division=0),
        "test_F1": f1_score(y_test, test_pred, zero_division=0),
        "test_ROC_AUC": roc_auc_score(y_test, test_prob),
    }
    return row, test_pred, test_prob

In [ ]:
if DEEPCHEM_AVAILABLE:
    tox_models = {
        "Random Forest": RandomForestClassifier(
            n_estimators=500,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
        ),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=500,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
        ),
    }

    tox_rows = []
    tox_predictions = {}

    for model_name, model in tox_models.items():
        row, pred, prob = evaluate_classifier(
            model,
            X_train_tox,
            y_train_tox,
            X_valid_tox,
            y_valid_tox,
            X_test_tox,
            y_test_tox,
        )
        row["model"] = model_name
        row["task"] = task_name
        tox_rows.append(row)
        tox_predictions[model_name] = {"pred": pred, "prob": prob}

    tox_results_df = pd.DataFrame(tox_rows)[[
        "model", "task",
        "valid_accuracy", "valid_precision", "valid_recall", "valid_F1", "valid_ROC_AUC",
        "test_accuracy", "test_precision", "test_recall", "test_F1", "test_ROC_AUC",
    ]]

    display(tox_results_df)

In [ ]:
if DEEPCHEM_AVAILABLE:
    best_tox_name = tox_results_df.sort_values("test_ROC_AUC", ascending=False).iloc[0]["model"]
    best_tox_pred = tox_predictions[best_tox_name]["pred"]
    best_tox_prob = tox_predictions[best_tox_name]["prob"]

    print("Best Tox21 classifier:", best_tox_name)
    print(classification_report(y_test_tox, best_tox_pred, zero_division=0))

    ConfusionMatrixDisplay.from_predictions(y_test_tox, best_tox_pred, display_labels=["Inactive", "Active"])
    plt.title(f"Tox21 {task_name}: confusion matrix")
    plt.show()

    RocCurveDisplay.from_predictions(y_test_tox, best_tox_prob)
    plt.title(f"Tox21 {task_name}: ROC curve")
    plt.show()

## Chapter 7 — Version 3C: DeepChem + PyTorch neural network

This section keeps DeepChem for loading, ECFP featurisation and scaffold splitting, but uses a custom PyTorch MLP for training.

This avoids depending on DeepChem's TensorFlow/Keras-backed neural-network classes, which may be unstable in some modern notebook environments.

In [ ]:
if DEEPCHEM_AVAILABLE:
    # Reuse the DeepChem ESOL arrays from Version 3A.
    X_train_nn = X_train_dc.astype(np.float32)
    y_train_nn = y_train_dc.astype(np.float32)
    X_valid_nn = X_valid_dc.astype(np.float32)
    y_valid_nn = y_valid_dc.astype(np.float32)
    X_test_nn = X_test_dc.astype(np.float32)
    y_test_nn = y_test_dc.astype(np.float32)

    dc_train_ds = MoleculeRegressionDataset(X_train_nn, y_train_nn)
    dc_valid_ds = MoleculeRegressionDataset(X_valid_nn, y_valid_nn)
    dc_test_ds = MoleculeRegressionDataset(X_test_nn, y_test_nn)

    dc_train_loader = DataLoader(dc_train_ds, batch_size=64, shuffle=True)
    dc_valid_loader = DataLoader(dc_valid_ds, batch_size=64, shuffle=False)
    dc_test_loader = DataLoader(dc_test_ds, batch_size=64, shuffle=False)

    dc_nn_model = FingerprintMLPRegressor(input_dim=X_train_nn.shape[1]).to(v1_device)
    dc_nn_criterion = nn.MSELoss()
    dc_nn_optimizer = torch.optim.AdamW(dc_nn_model.parameters(), lr=5e-4, weight_decay=1e-3)

    print(dc_nn_model)

In [ ]:
if DEEPCHEM_AVAILABLE:
    n_epochs = 100
    patience = 15
    best_valid_loss = np.inf
    best_state = None
    patience_counter = 0

    dc_nn_train_losses = []
    dc_nn_valid_losses = []

    for epoch in range(n_epochs):
        dc_nn_model.train()
        running_loss = 0.0

        for batch_X, batch_y in dc_train_loader:
            batch_X = batch_X.to(v1_device)
            batch_y = batch_y.to(v1_device)

            dc_nn_optimizer.zero_grad()
            preds = dc_nn_model(batch_X)
            loss = dc_nn_criterion(preds, batch_y)
            loss.backward()
            dc_nn_optimizer.step()

            running_loss += loss.item() * batch_X.size(0)

        train_loss = running_loss / len(dc_train_ds)
        valid_loss = evaluate_mse_loss(dc_nn_model, dc_valid_loader, v1_device)

        dc_nn_train_losses.append(train_loss)
        dc_nn_valid_losses.append(valid_loss)

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = {k: v.detach().cpu().clone() for k, v in dc_nn_model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1:03d} | Train MSE {train_loss:.4f} | Valid MSE {valid_loss:.4f}")

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    if best_state is not None:
        dc_nn_model.load_state_dict(best_state)

In [ ]:
if DEEPCHEM_AVAILABLE:
    plt.figure(figsize=(7, 5))
    plt.plot(dc_nn_train_losses, label="Train")
    plt.plot(dc_nn_valid_losses, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("MSE loss")
    plt.title("Version 3C DeepChem ECFP + PyTorch MLP")
    plt.legend()
    plt.show()

    y_train_true_dc, y_train_pred_dc = predict_regression(dc_nn_model, dc_train_loader, v1_device)
    y_valid_true_dc, y_valid_pred_dc = predict_regression(dc_nn_model, dc_valid_loader, v1_device)
    y_test_true_dc, y_test_pred_dc = predict_regression(dc_nn_model, dc_test_loader, v1_device)

    dc_nn_metrics_df = pd.DataFrame([
        {"split": "train", **regression_metrics(y_train_true_dc, y_train_pred_dc)},
        {"split": "valid", **regression_metrics(y_valid_true_dc, y_valid_pred_dc)},
        {"split": "test", **regression_metrics(y_test_true_dc, y_test_pred_dc)},
    ])
    display(dc_nn_metrics_df)

In [ ]:
if DEEPCHEM_AVAILABLE:
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test_true_dc, y_test_pred_dc, alpha=0.7)
    min_value = min(y_test_true_dc.min(), y_test_pred_dc.min())
    max_value = max(y_test_true_dc.max(), y_test_pred_dc.max())
    plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
    plt.xlabel("Actual logS")
    plt.ylabel("Predicted logS")
    plt.title("Version 3C: predicted vs actual")
    plt.show()

    residuals = y_test_true_dc - y_test_pred_dc
    plt.figure(figsize=(7, 5))
    plt.scatter(y_test_pred_dc, residuals, alpha=0.7)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted logS")
    plt.ylabel("Residual: actual - predicted")
    plt.title("Version 3C residual plot")
    plt.show()

## Chapter 8 — Model comparison and project interpretation

Use this section to compare the models you actually ran.

Key interpretation:

- Version 1 is the simplest neural-network molecular baseline.
- Version 2 is the graph-learning direction and is more chemically structural.
- Version 3A is a strong non-neural benchmark.
- Version 3B teaches molecular classification on real toxicity tasks.
- Version 3C shows how to combine DeepChem data handling with custom PyTorch models.

For small-to-medium molecular datasets, tree models on ECFP can outperform MLPs. That is a valid result, not a failure. Deep learning becomes more useful with larger datasets, better representations, pretraining, graph models and multi-task labels.

In [ ]:
comparison_rows = []

if "v1_metrics_df" in globals():
    test_row = v1_metrics_df[v1_metrics_df["split"] == "test"].iloc[0].to_dict()
    comparison_rows.append({"model": "Version 1: Morgan MLP", **{k: test_row[k] for k in ["MAE", "RMSE", "R2"]}})

if "baseline_df" in globals():
    for _, row in baseline_df.iterrows():
        comparison_rows.append({
            "model": f"Version 3A: {row['model']}",
            "MAE": row["test_MAE"],
            "RMSE": row["test_RMSE"],
            "R2": row["test_R2"],
        })

if "dc_nn_metrics_df" in globals():
    test_row = dc_nn_metrics_df[dc_nn_metrics_df["split"] == "test"].iloc[0].to_dict()
    comparison_rows.append({"model": "Version 3C: DeepChem + PyTorch MLP", **{k: test_row[k] for k in ["MAE", "RMSE", "R2"]}})

if comparison_rows:
    comparison_df = pd.DataFrame(comparison_rows).sort_values("RMSE").reset_index(drop=True)
    display(comparison_df)
else:
    print("Run Version 1, Version 3A or Version 3C first to build a regression comparison table.")

## Chapter 9 — Extension ideas

Good next upgrades:

1. Add bond features to the GNN.
2. Replace GCN with GIN or MPNN.
3. Convert Version 2 from classification to exact logS regression.
4. Add scaffold split to the manual PyTorch workflow.
5. Pretrain on larger datasets such as ChEMBL or ZINC.
6. Fine-tune on perfume/odour descriptor data.
7. Convert perfume labels into multi-label targets such as sweet, floral, citrus, woody, green and musk.

Final memory sentence:

```text
Version 1 learns from molecular vectors; Version 2 learns from atom-bond graphs; Version 3 uses DeepChem for benchmark-style molecular ML.
```